# Lesson 16 — 连接前先判断网络状态

这节课仍然延续桌面状态提醒器，但数据首次穿过**真实本机网络**。不需要 ESP32，也不需要 Python 基础：下面代码格是完整 C++ 程序。

## 我们要解决什么问题？

设备在网络尚未可用时，为什么不应该直接无限重连服务器？

## 最小知识

Wi-Fi 关联、DHCP、DNS 和 TCP 是不同阶段。没有 ESP32 射频时，本课用一个明确标记的 Wi-Fi 状态模型控制是否允许连接；一旦允许，`localhost` 的 DNS 解析和 TCP PING/PONG 都由 C++ 真实执行。

运行初始化格后，所有网络实验只绑定 `127.0.0.1`。Notebook 面板是真实客户端；业务逻辑、协议响应和状态都来自当前 C++ 代码格。

In [ ]:
# 运行支架：只执行一次。你写的网络逻辑仍是 C++，不需要学习 Python。
from pathlib import Path
project = next((p for p in [Path.cwd(), *Path.cwd().parents]
                if (p / "tools/notebook_setup.py").is_file()), None)
if project is None:
    raise RuntimeError("请用 VS Code 打开 esp32-hands-on-academy 文件夹")
get_ipython().run_line_magic("run", str(project / "tools/notebook_setup.py"))
from network_lab import register
register()


## 1. 第一个可运行实验

分别点击断开与已连接。断开状态应只输出 WAIT_FOR_WIFI；连接状态则由当前 C++ 解析 `localhost` 为 `127.0.0.1` 并连接真实本机 fixture。

运行下一格。面板的端口由 C++ 程序通过 `ACADEMY_READY` 报告；不要手填固定端口。

In [ ]:
%%network_lab l16-guided dns guided pass
#include "academy/net/socket.hpp"
#include <iostream>

int main() {
    academy::net::SocketSystem sockets;
    if (!academy::net::env_flag("ACADEMY_WIFI_CONNECTED")) {
        std::cout << "WAIT_FOR_WIFI\n";
        return 0;
    }
    const auto address = academy::net::resolve_ipv4("localhost");
    const auto reply = academy::net::request_line("localhost", academy::net::env_port("ACADEMY_PORT"), "PING\n");
    std::cout << "RESOLVED " << address << ' ' << reply;
    return reply == "PONG\n" ? 0 : 1;
}


## 2. 修改一次

把下一格中一个响应文本或状态初值改掉，再点击面板操作，观察改变从 C++ 穿过真实 localhost 连接回到输出。重新运行本格会终止旧进程并启动新进程。

In [ ]:
%%network_lab l16-modify dns
#include "academy/net/socket.hpp"
#include <iostream>

int main() {
    academy::net::SocketSystem sockets;
    if (!academy::net::env_flag("ACADEMY_WIFI_CONNECTED")) {
        std::cout << "WAIT_FOR_WIFI\n";
        return 0;
    }
    const auto address = academy::net::resolve_ipv4("localhost");
    const auto reply = academy::net::request_line("localhost", academy::net::env_port("ACADEMY_PORT"), "PING\n");
    std::cout << "RESOLVED " << address << ' ' << reply;
    return reply == "PONG\n" ? 0 : 1;
}


## 3. 故意弄坏

下面代码可编译，但**自动检查应失败**。先运行，点击“播放自动检查”，再根据红色结果解释问题。失败来自真实网络往返后的行为差异，不是 Notebook 预设动画。

In [ ]:
%%network_lab l16-broken dns exercise fail
#include "academy/net/socket.hpp"
#include <iostream>

int main() {
    academy::net::SocketSystem sockets;
    // 错误：无论模型是否已连接，都尝试连接服务。
    const auto address = academy::net::resolve_ipv4("localhost");
    const auto reply = academy::net::request_line("localhost", academy::net::env_port("ACADEMY_PORT"), "PING\n");
    std::cout << "RESOLVED " << address << ' ' << reply;
    return 0;
}


## 4. 自己完成

在下方 C++ 中完成 TODO。先手动操作，再点“播放自动检查”。测试只编译这一格，不会读取 `solution/`。需要时再看 [分层 Hint](../lessons/16-connection-state-and-dns/hints.md)。

In [ ]:
%%network_lab l16-exercise dns exercise
#include "academy/net/socket.hpp"
#include <iostream>

int main() {
    academy::net::SocketSystem sockets;
    // TODO: 未连接时输出 WAIT_FOR_WIFI 并 return 0。
    // 已连接时 resolve localhost，发送 PING\n，输出 RESOLVED <ip> PONG。
    std::cout << "TODO\n";
    return 0;
}


## 5. 你刚刚真正学到了什么？

你已经让提醒器通过真实 localhost 网络与另一个进程通信，并使用协议契约验证状态。ESP32 上应把 Wi-Fi event、IP obtained event、DNS 失败和指数退避重连组织成状态机。不要将 host 的 `ACADEMY_WIFI_CONNECTED` 当作真实无线验证。

完成尝试后才查看 [Solution Notebook](solutions/16-connection-state-and-dns-solution.ipynb)。